# Session 5 — A Model That Helps You Prompt

### AI Image Studio · Session 5 of 8

### This is the last notebook session. Next time we build the real project.

Two ideas today, and the second one is the most important in the whole course.

**First:** you can use one AI model to improve the input to another one. A text model that
is good at writing can take your five-word idea and turn it into the kind of detailed
description we learned to write in Session 3 — automatically, every time.

**Second:** the code we've been writing is not tied to any one AI company. Today you're
going to change **one line** and watch the same program run against a completely different
company's model — or against a model running on this laptop, with the wifi turned off.

And then we're going to find the place where that trick **stops working**, and talk about
why, because an abstraction you don't know the limits of is a trap.

## TODAY:

- **Part A:** a text model, and what a system prompt is
- **Part B:** `improve_prompt()` — and honestly asking whether it helps
- **Part C:** the one-line provider swap
- **Part D:** where the abstraction leaks

In [ ]:
# imports

import os
import re
import base64
from pathlib import Path
from datetime import date
from dotenv import load_dotenv
from IPython.display import Image, display
from langchain.chat_models import init_chat_model
from langchain_core.messages import SystemMessage, HumanMessage

In [ ]:
# Setup

load_dotenv(override=True)
api_key = os.getenv("OPENAI_API_KEY")
print(f"✅ Key loaded, begins {api_key[:8]}" if api_key else "❌ No key found - check your .env")

CHAT_MODEL = "openai:gpt-4.1-mini"
IMAGE_TOOL = {"type": "image_generation", "quality": "low"}

In [ ]:
# Everything from sessions 2-4, brought forward one last time

image_model = init_chat_model(CHAT_MODEL).bind_tools([IMAGE_TOOL])
GALLERY = Path("gallery")
SIZES = {"square": "1024x1024", "wide": "1536x1024", "tall": "1024x1536"}
STYLES = {
    "photo": "photorealistic, natural lighting, shot on 35mm film, shallow depth of field",
    "watercolour": "watercolour painting, soft edges, muted colours, visible paper texture",
    "pixel": "16-bit pixel art, limited colour palette, crisp pixels, retro game sprite",
    "cinematic": "cinematic film still, dramatic lighting, wide aspect, moody atmosphere",
    "sketch": "rough pencil sketch, black and white, visible construction lines",
    "cute": "cute flat vector illustration, bold outlines, bright friendly colours",
}


def extract_image_bytes(response):
    """Find the image inside a model's response and return it as raw bytes."""
    for block in response.content_blocks:
        if block["type"] == "image":
            return base64.b64decode(block["base64"])
    raise ValueError("The model didn't return an image. What did it say instead?")


def make_image(prompt, style_name="photo", size="square"):
    """Generate an image using a named style, and return it as bytes."""
    response = image_model.invoke(f"{prompt}. {STYLES[style_name]}. Image size: {SIZES[size]}.")
    return extract_image_bytes(response)


def slugify(text, max_words=5):
    """Turn a prompt into something safe to use as a filename."""
    text = re.sub(r"[^a-z0-9 ]", "", text.lower())
    return "-".join(text.split()[:max_words]) or "image"


def save_image(image_bytes, prompt, style_name):
    """Save an image into the gallery folder with a descriptive filename."""
    GALLERY.mkdir(exist_ok=True)
    path = GALLERY / f"{date.today()}_{slugify(prompt)}_{style_name}.png"
    path.write_bytes(image_bytes)
    return path

## PART A: A text model, and the system prompt

In Session 1 we made a text model and asked it a question. Now we're going to give it a
**job**, and that's a different thing.

In [ ]:
# A plain text model. No image tool bound to it - this one only writes.

text_model = init_chat_model(CHAT_MODEL)

### Two kinds of message

When you talk to a chat model, your message has a **role**. The two that matter here:

- **System** — who the model is and what its job is. Set once, by *you*, the programmer.
  The user never sees it and usually can't change it.
- **Human** — what the user actually typed.

The separation is the point. It's how ChatGPT can be helpful and polite while you type
whatever you like at it: there's a system message you never see, telling it how to behave.

In [ ]:
# Compare. First, with no system message at all:

text_model.invoke("a cat on a windowsill").text

The model had no idea what we wanted, so it guessed. It probably wrote a little description,
or asked what we meant. Now let's tell it what its job is:

In [ ]:
messages = [
    SystemMessage(content="You turn short ideas into vivid image-generation prompts. Reply with the prompt only."),
    HumanMessage(content="a cat on a windowsill"),
]

text_model.invoke(messages).text

Same model. Same user message. Completely different behaviour — because we told it what it
was for.

<div style="border-left: 6px solid #06c; background: #f2f7ff; padding: 12px 16px; margin: 12px 0;">
<h3 style="color:#06c; margin-top:0;">🔄 What that list is</h3>
<p style="color:#06c; margin-bottom:0;">
<code>[SystemMessage(...), HumanMessage(...)]</code> is a Python <b>list</b> of two objects
— square brackets, comma-separated, exactly like a JavaScript array.<br/><br/>
<code>SystemMessage(content="...")</code> is creating an object, the way you'd write
<code>new SystemMessage({content: "..."})</code> in JavaScript. Python doesn't use
<code>new</code>; you just call the class like a function.
</p>
</div>

## PART B: `improve_prompt()`

Now we make it a proper part of our studio. And note where the system prompt lives — in a
CAPITALS constant, on its own, not buried inside the function.

A system prompt is something you will edit and re-edit dozens of times while tuning
behaviour. Keeping it at the top, where you can see all of it at once, makes that possible.

In [ ]:
PROMPT_IMPROVER_SYSTEM = """
You are a prompt engineer for an image generation model.
The user gives you a short, plain idea. You rewrite it as one vivid sentence
that an image model can work with.

Include: the subject, what is happening, the lighting, and the mood.
Do NOT add an art style - the studio applies that separately.
Do NOT explain yourself, do not use quotation marks, do not add a preamble.
Reply with the improved prompt and nothing else.
"""

In [ ]:
def improve_prompt(rough_prompt):
    """
    Use a text model to expand a short idea into a detailed image prompt.

    This is the same thing we learned to do by hand in Session 3 - add lighting, mood
    and detail - except a model does it, so it happens every time instead of only when
    we can be bothered.

    Returns the improved prompt as a plain string.
    """
    response = text_model.invoke([
        SystemMessage(content=PROMPT_IMPROVER_SYSTEM),
        HumanMessage(content=rough_prompt),
    ])
    return response.text.strip()

In [ ]:
# Try it on something deliberately lazy

improve_prompt("a dragon")

In [ ]:
# And a few more. Notice it adds different things depending on the subject.

for idea in ["a school bus", "my bedroom", "a robot chef"]:
    print(f"{idea:15} ->  {improve_prompt(idea)}\n")

### Does it actually help? Let's check, not assume.

This is the part that matters. It is extremely easy to add a clever feature and simply
*believe* it improved things. Test it: same idea, one with, one without.

In [ ]:
rough = "a dragon"

# Without the improver
display(Image(make_image(rough, "cinematic")))

In [ ]:
# With the improver

improved = improve_prompt(rough)
print(improved)
display(Image(make_image(improved, "cinematic")))

<div style="border-left: 6px solid #900; background: #fff4f4; padding: 12px 16px; margin: 12px 0;">
<h3 style="color:#900; margin-top:0;">🎯 Now try it yourself — and be honest</h3>
<p style="color:#900; margin-bottom:0;">
Run the comparison on <b>three</b> of your own ideas. For each one, decide which image you
actually prefer.<br/><br/>
Here's the important bit: <b>sometimes the improver will make it worse.</b> It will add
details you didn't want, or drag everything toward the same generic "epic golden hour" look.
When that happens, say so. Noticing that your clever feature isn't helping is a genuine
result, not a failure — and it's the reason we're going to make the improver
<i>optional</i> in the final project rather than always-on.
</p>
</div>

In [ ]:
# Your comparison. Try it on something where you have a clear picture in your head.

my_idea = ""

print("WITHOUT:")
display(Image(make_image(my_idea, "photo")))

In [ ]:
print("WITH:")
improved = improve_prompt(my_idea)
print(improved)
display(Image(make_image(improved, "photo")))

## PART C: The one-line provider swap

Here is the payoff for everything we've done through LangChain instead of through one
company's own library.

Look at how we made both of our models:

```python
text_model  = init_chat_model(CHAT_MODEL)
image_model = init_chat_model(CHAT_MODEL).bind_tools([IMAGE_TOOL])
```

Neither line mentions OpenAI. The company name only exists inside `CHAT_MODEL`, which is
a string, in one place, at the top of the notebook.

### What you can put in that string

```python
CHAT_MODEL = "openai:gpt-4.1-mini"                    # what we've used all course
CHAT_MODEL = "anthropic:claude-haiku-4-5-20251001"    # a different company
CHAT_MODEL = "google-genai:gemini-2.5-flash"          # another one
CHAT_MODEL = "ollama:llama3.2"                        # running on THIS laptop. No key. No internet.
```

Everything after that line — `improve_prompt`, `PROMPT_IMPROVER_SYSTEM`, the
`SystemMessage`/`HumanMessage` list, `.invoke()`, `.text` — stays **exactly the same**.

Each provider needs its own package installed (`langchain-anthropic`, `langchain-ollama`
and so on) and, except for Ollama, its own key in `.env`. But not one line of *your* code
changes.

In [ ]:
# Let's prove it. This creates a second text model from a different string.
#
# Uncomment ONE of these. Ollama needs Ollama installed and running locally
# (`ollama run llama3.2` in a terminal, once). The others need a key in your .env.

# other_model = init_chat_model("ollama:llama3.2")
# other_model = init_chat_model("anthropic:claude-haiku-4-5-20251001")
# other_model = init_chat_model("google-genai:gemini-2.5-flash")

other_model = text_model   # delete this line once you've uncommented one above

In [ ]:
# The exact same function body, pointed at a different company's model.
# Read it carefully: there is nothing OpenAI-shaped anywhere in here.

def improve_prompt_with(model, rough_prompt):
    """Same as improve_prompt, but you choose which model does the work."""
    response = model.invoke([
        SystemMessage(content=PROMPT_IMPROVER_SYSTEM),
        HumanMessage(content=rough_prompt),
    ])
    return response.text.strip()

In [ ]:
# Same idea, two different models. Compare how they write.

idea = "a robot learning to paint"

print("Model A:", improve_prompt_with(text_model, idea))
print()
print("Model B:", improve_prompt_with(other_model, idea))

<div style="border-left: 6px solid #181; background: #f3fbf3; padding: 12px 16px; margin: 12px 0;">
<h3 style="color:#181; margin-top:0;">🌍 Where you'll see this in the real world</h3>
<p style="color:#181; margin-bottom:0;">
This is called <b>avoiding vendor lock-in</b>, and it is a genuine, expensive, boardroom-level
problem for real companies.<br/><br/>
Companies have written a hundred thousand lines against one AI provider, and then that
provider raises its prices, or a competitor releases something twice as good, or their
legal team decides the data can't leave the country any more — and switching costs them
six months.<br/><br/>
You've just avoided that for the price of writing <code>init_chat_model("openai:...")</code>
instead of <code>OpenAI()</code>. That is the entire trick, and it cost you nothing.
</p>
</div>

## PART D: Where the abstraction leaks

Now the honest part, and the bit most tutorials skip.

Try the swap on the **image** model:

```python
image_model = init_chat_model("ollama:llama3.2").bind_tools([IMAGE_TOOL])
```

It will not work. Not because we wrote it badly — because **Ollama has no image-generation
model behind that interface at all.** There is nothing there to call. The same is true of
most text-only models at every provider.

So our studio splits into two halves with different freedoms:

| Part | Can you swap the provider? |
|---|---|
| `improve_prompt` — the text half | **Yes.** OpenAI, Anthropic, Google, or locally on this laptop |
| `make_image` — the image half | **Only between providers that can actually draw** (OpenAI, Google) |

### Why this is worth understanding

An abstraction is a promise that several different things can be treated as the same thing.
That promise holds exactly as far as the things really *are* similar, and no further.

`init_chat_model` genuinely unifies "send text, get text back", because every chat model
does that. It cannot unify "draw me a picture", because not every chat model can.

The useful skill isn't knowing this one fact about LangChain. It's the habit of asking,
about **any** abstraction you're handed: *what is it actually promising, and where does that
promise run out?* Code that assumes an abstraction is total will break in a confusing way at
the exact moment you're least expecting it.

<div style="border-left: 6px solid #900; background: #fff4f4; padding: 12px 16px; margin: 12px 0;">
<h3 style="color:#900; margin-top:0;">🎯 Now try it yourself</h3>
<p style="color:#900; margin-bottom:0;">
Change <code>CHAT_MODEL</code> at the top of this notebook to a different provider, restart
the kernel, and run the whole notebook from the top.<br/><br/>
Write down: <b>(a)</b> what you had to change besides that one line (installing a package?
adding a key?), and <b>(b)</b> whether the improved prompts read differently.<br/><br/>
Then put it back to <code>openai:gpt-4.1-mini</code> before next session.
</p>
</div>

<div style="border-left: 6px solid #f71; background: #fff7f0; padding: 12px 16px; margin: 12px 0;">
<h3 style="color:#f71; margin-top:0;">🚀 Extra challenge</h3>
<p style="color:#f71; margin-bottom:0;">
Write a <b>second</b> system prompt with a completely different personality — one that turns
every idea into something absurd, or noir, or unbearably cute — and a function that picks
between them.<br/><br/>
Then look at what you've built: a program whose behaviour is controlled by which paragraph
of English you feed it. That's a genuinely new kind of programming, and it's only about
three years old.
</p>
</div>

## Where we got to

- System messages give a model a job; the job lives in a constant, not buried in a function
- `improve_prompt()` — one model preparing the input for another
- We tested whether it helps rather than assuming, and found it sometimes doesn't
- The one-line provider swap, and why companies pay real money for that property
- Where the abstraction leaks, and why knowing that is part of using it properly

## Next session — the big one

**We leave the notebook.**

Everything you've written over five sessions — `make_image`, `STYLES`, `slugify`,
`save_image`, `improve_prompt` — gets moved out of these cells and into real Python files
that you can run from a terminal, hand to a friend, and keep.

Nothing new to learn. Everything to reorganise. Bring all five notebooks.